# Base year (2025) validation

This notebook validates the results of PyPSA-Earth for the base year (2025) against available Australian datasets. The validation covers the following aspect of the electricity system:

- **Installed Generation Capacity**

For installed capacity, the reference dataset is the AEMO Generator Information dataset (2026 release), filtered to approximate the active generation fleet up to the end of 2025:
- Commitment Status ∈ {In Service, Announced Withdrawal, Committed}
- Full Commercial Use Date ≤ 2025-12-31 or missing
- Closure Date > 2025-12-31 or missing

Note: The AEMO dataset reflects the National Electricity Market (NEM), not the entirety of Australia, while PyPSA-Earth represents the whole country. Therefore, this comparison is indicative and not a strict national validation.

## 1. Setup and Data Loading

*This section handles the initial setup, including importing necessary libraries and loading the solved PyPSA-Earth networks.*

### 1.1 Loading libraries

We begin by importing the necessary libraries for data handling, analysis, and visualization. These include PyPSA for power system analysis, pandas and numpy for data manipulation, geopandas for spatial data, and seaborn/matplotlib for plotting.

---

In [ ]:
# Install required packages if not already installed
# Uncomment the line below to install packages
# Note: This line is commented out to prevent installation during code execution.

# !pip install numpy pandas plotly pycountry matplotlib seaborn -qq

In [ ]:
import os
import pypsa
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

import pycountry
import warnings

warnings.filterwarnings("ignore")

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

### 1.2 Loading Files

PyPSA-Earth solved networks for the base year are uploaded here, as well as the relevant EIA and Ember datasets. These datasets provide reference values for demand, generation, and installed capacity, which will be used for validation.

In [ ]:
DATA_DIR = "data/validation_data/"

aemo_file = os.path.join(DATA_DIR, "NEM Generation Information Jan 2026.xlsx")

In [ ]:
results_dir = "../pypsa-earth/results/postnetworks"
network_path = os.path.join(
    results_dir,
    "elec_s_10_ec_lv1_Co2L-3h_3h_2025_0.071_AB_0export.nc",
)
base_network = pypsa.Network(network_path)

## 1.3 Loading input data

### 1.3.1. Installed capacity

## Data source: AEMO Generator Information (Jan 2026)

This notebook uses the AEMO Generator Information dataset.

Due to licensing restrictions, the file is **not included in the repository**.

Download it manually from:
https://www.aemo.com.au/energy-systems/electricity/national-electricity-market-nem/nem-forecasting-and-planning/forecasting-and-planning-data/generation-information

and place it in:

`notebooks/data/validation_data/NEM Generation Information Jan 2026.xlsx`

In [ ]:
aemo_file = Path("data/validation_data/NEM Generation Information Jan 2026.xlsx")

if not aemo_file.exists():
    raise FileNotFoundError(f"""
AEMO dataset not found.

Please download:
https://www.aemo.com.au/energy-systems/electricity/national-electricity-market-nem/nem-forecasting-and-planning/forecasting-and-planning-data/generation-information

and place it here:
{aemo_file}
""")

In [ ]:
# Read AEMO generator information
aemo = pd.read_excel(
    aemo_file,
    sheet_name="Generator Information",
    header=3,
)

# Clean column names
aemo.columns = (
    aemo.columns.astype(str)
    .str.replace("\n", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Include plants relevant up to end of 2025:
# - In Service
# - Announced Withdrawal
# - Committed
aemo_2025 = aemo[
    aemo["Commitment Status"].isin(["In Service", "Announced Withdrawal", "Committed"])
].copy()

# Parse dates
aemo_2025["Full Commercial Use Date"] = pd.to_datetime(
    aemo_2025["Full Commercial Use Date"],
    errors="coerce",
)

aemo_2025["Closure Date"] = pd.to_datetime(
    aemo_2025.get("Closure Date"),
    errors="coerce",
)

# Keep plants active up to end of 2025:
# - commissioned before or during 2025
# - not closed before end of 2025
cutoff_date = "2025-12-31"

aemo_2025 = aemo_2025[
    (
        aemo_2025["Full Commercial Use Date"].isna()
        | (aemo_2025["Full Commercial Use Date"] <= cutoff_date)
    )
    & (aemo_2025["Closure Date"].isna() | (aemo_2025["Closure Date"] > cutoff_date))
].copy()

# Build a refined technology class using both Technology Type and Technology Detail
aemo_2025["tech_class"] = aemo_2025["Technology Type"].copy()

# Hydro split: Pumped Hydro -> PHS, Dam + Run of River -> Hydro
aemo_2025.loc[
    (aemo_2025["Technology Type"] == "Hydro")
    & (aemo_2025["Technology Detail"] == "Pumped Hydro"),
    "tech_class",
] = "PHS"

aemo_2025.loc[
    (aemo_2025["Technology Type"] == "Hydro")
    & (aemo_2025["Technology Detail"].isin(["Dam", "Run of River"])),
    "tech_class",
] = "Hydro"

# Other split:
# - Biomass stays Biomass
# - everything else under Other -> Oil
aemo_2025.loc[
    (aemo_2025["Technology Type"] == "Other")
    & (aemo_2025["Technology Detail"] == "Biomass"),
    "tech_class",
] = "Biomass"

aemo_2025.loc[
    (aemo_2025["Technology Type"] == "Other")
    & (aemo_2025["Technology Detail"] != "Biomass"),
    "tech_class",
] = "Oil"

# Aggregate installed capacity from AEMO
aemo_capacity = (
    aemo_2025.groupby("tech_class")["Agg Nameplate Capacity (MW AC)"]
    .sum()
    .sort_values(ascending=False)
)

# Map AEMO technologies to notebook categories
aemo_mapping = {
    "Battery Storage": "Battery storage",
    "Bioenergy": "Biomass",
    "Biomass": "Biomass",
    "Gas Turbine": "Natural gas",
    "Hydro": "Hydro",
    "PHS": "PHS",
    "Solar PV": "Solar",
    "Wind": "Wind",
    "Diesel Engine": "Oil",
    "Reciprocating Engine": "Oil",
    "Steam Turbine": "Coal",
    "Oil": "Oil",
}

aemo_capacity = aemo_capacity.rename(index=aemo_mapping).groupby(level=0).sum()

# Convert MW -> GW
aemo_capacity_df = (aemo_capacity / 1000).round(2).to_frame("AEMO Gen Info (2026)")

In [ ]:
country_code = "AU"
horizon = 2025

### 1.3.2. Electricity generation

In [ ]:
reference_file = Path(
    "data/validation_data/Australian Energy Statistics 2025 Table O.xlsx"
)

if not reference_file.exists():
    raise FileNotFoundError(f"""
Australian Energy Statistics dataset not found.


Please download:
https://www.energy.gov.au/publications/australian-energy-statistics-table-o-electricity-generation-fuel-type-2023-24-and-2024#:~:text=Australian%20Energy%20Statistics%2C%20Table%20O%20Electricity%20generation%20by%20fuel%20type%202023%2D24%20and%202024%20(XLSX%2097.84%20KB)

and place it here:
{reference_file}
""")

In [ ]:
reference_df = pd.read_excel(
    reference_file,
    sheet_name="AUS FY",
    header=4,
)

reference_generation = (
    reference_df[["Unnamed: 1", "2023-24"]]
    .rename(
        columns={
            "Unnamed: 1": "technology",
            "2023-24": "generation_gwh",
        }
    )
    .dropna(subset=["technology"])
)

reference_generation = reference_generation[
    reference_generation["technology"].isin(
        [
            "Black coal",
            "Brown coal",
            "Natural gas",
            "Oil products",
            "Hydro",
            "Wind",
            "Large-scale solar PV",
            "Small-scale solar PV",
            "Bagasse, wood",
            "Biogas",
        ]
    )
].copy()

reference_generation["carrier"] = reference_generation["technology"].replace(
    {
        "Black coal": "coal",
        "Brown coal": "coal",
        "Natural gas": "gas",
        "Oil products": "oil",
        "Hydro": "hydro",
        "Wind": "wind",
        "Large-scale solar PV": "solar",
        "Small-scale solar PV": "solar",
        "Bagasse, wood": "biomass",
        "Biogas": "biomass",
    }
)

reference_generation = (
    reference_generation.groupby("carrier")["generation_gwh"].sum() / 1000
)

reference_generation.name = "Reference"

---

## 2. Installed capacity

This section validates the installed generation capacities by technology.
The PyPSA network is compared against AEMO Generator Information filtered to:
- Commitment Status = In Service
- Full Commercial Use Date <= 2025-12-31 or empty

Note: this AEMO source reflects NEM generator data rather than all-Australia capacity.

In [ ]:
# Use AEMO as the reference dataset for installed capacity
installed_capacity_ref = aemo_capacity_df.copy()

In [ ]:
gen_carriers = {
    "onwind",
    "offwind-ac",
    "offwind-dc",
    "solar",
    "solar rooftop",
    "csp",
    "nuclear",
    "geothermal",
    "ror",
    "PHS",
    "hydro",
    "solid biomass",
}
link_carriers = {
    "OCGT",
    "CCGT",
    "coal",
    "oil",
    "biomass",
    "urban central solid biomass CHP",
    "gas CHP",
    "battery discharger",
}

# Generators
gen = base_network.generators.copy()
gen["carrier"] = gen["carrier"].replace(
    {"offwind-ac": "offwind", "offwind-dc": "offwind"}
)
gen = gen[gen.carrier.isin(gen_carriers)]
gen_totals = gen.groupby("carrier")["p_nom_opt"].sum()

# Storage
sto = base_network.storage_units.copy()
sto = sto[sto.carrier.isin(gen_carriers)]
sto_totals = sto.groupby("carrier")["p_nom_opt"].sum()

# Links (output side scaled by efficiency)
links = base_network.links.copy()
mask = (
    links.efficiency.notnull()
    & (links.p_nom_opt > 0)
    & links.carrier.isin(link_carriers)
)
links = links[mask]
links_totals = links.groupby("carrier").apply(
    lambda df: (df["p_nom_opt"] * df["efficiency"]).sum()
)

# Combine all
all_totals = pd.concat([gen_totals, sto_totals, links_totals])
all_totals = all_totals.groupby(all_totals.index).sum()  # Merge duplicates
all_totals = all_totals[all_totals > 0] / 1e3

In [ ]:
pypsa_cap = (
    all_totals.rename(
        {
            "onwind": "Wind",
            "offwind": "Wind",
            "solar rooftop": "Solar rooftop",
            "solar": "Solar",
            "ror": "Hydro",
            "geothermal": "Geothermal",
            "nuclear": "Nuclear",
            "hydro": "Hydro",
            "OCGT": "Natural gas",
            "CCGT": "Natural gas",
            "oil": "Oil",
            "coal": "Coal",
            "lignite": "Coal",
            "biomass": "Biomass",
            "urban central solid biomass CHP": "Biomass",
            "solid biomass": "Biomass",
            "battery discharger": "Battery storage",
        }
    )
    .to_frame("PyPSA-Earth results")
    .groupby(level=0)
    .sum()
)

# Skip only true zeros, without rounding first
pypsa_cap = pypsa_cap[pypsa_cap["PyPSA-Earth results"] > 0]

In [ ]:
fossil_fuels = ["Natural gas", "Oil", "Coal"]

# Sum fossil fuels, no KeyError if a carrier is missing
pypsa_fossil_fuels = pypsa_cap.reindex(fossil_fuels, fill_value=0).sum().iloc[0]
aemo_fossil_fuels = (
    installed_capacity_ref.reindex(fossil_fuels, fill_value=0).sum().iloc[0]
)

# Add aggregated fossil fuels row
pypsa_cap.loc["Fossil fuels"] = pypsa_fossil_fuels
installed_capacity_ref.loc["Fossil fuels"] = aemo_fossil_fuels

# Rename columns properly
pypsa_cap.columns = ["PyPSA-Earth (2025)"]
installed_capacity_ref.columns = ["AEMO Gen Info (2026)"]

# Concatenate datasets
installed_capacity_df = pd.concat(
    [pypsa_cap, installed_capacity_ref],
    axis=1,
).fillna(0)

# Display with 1 decimal and show zeros as N/A (visual only)
installed_capacity_df.style.format(lambda x: "N/A" if x == 0 else f"{x:.1f}")

In [ ]:
installed_capacity_df.plot(kind="bar", figsize=(12, 6), width=0.8)

plt.title("Installed Capacity Comparison")
plt.ylabel("Installed Capacity (GW)")
plt.xticks(rotation=45, ha="right")
plt.legend(loc="best")
plt.tight_layout()

## Notes:
- The AEMO Generator Information dataset represents only the **National Electricity Market (NEM)**, which includes eastern and southern Australian states (QLD, NSW, VIC, SA, TAS). 
In contrast, PyPSA-Earth models the entire Australian electricity system, including regions outside the NEM such as Western Australia and the Northern Territory.
As a result, some differences — in particular higher gas capacity in PyPSA — are expected and reflect additional generation assets outside the NEM rather than inconsistencies in the datasets.

## Source:
- AEMO: NEM January 2026 Generation Information, https://www.aemo.com.au/energy-systems/electricity/national-electricity-market-nem/nem-forecasting-and-planning/forecasting-and-planning-data/generation-information, 2026, Accessed Apr 20, 2026

## 3. Electricity generation

This section validates annual electricity generation by technology.

The PyPSA-AUS network is compared against Australian Energy Statistics 2025, Table O, sheet `AUS FY`, using financial year `2023-24`.

The reference data are aggregated to match the PyPSA carrier structure:
- Black coal + Brown coal: Coal
- Natural gas: Natural gas
- Oil products: Oil
- Large-scale solar PV + Small-scale solar PV: Solar
- Wind: Wind
- Hydro: Hydro
- Bagasse, wood + Biogas: Biomass

In [ ]:
# Compute PyPSA annual electricity generation
weights = base_network.snapshot_weightings.generators


def weighted_sum_timeseries(df, weights):
    """Return weighted annual sum for each column."""
    if df.empty:
        return pd.Series(dtype=float)

    return df.mul(weights, axis=0).sum()


# Generators
generators = base_network.generators.copy()

generators["carrier"] = generators["carrier"].replace(
    {
        "onwind": "Wind",
        "offwind-ac": "Wind",
        "offwind-dc": "Wind",
        "solar": "Solar",
        "solar rooftop": "Solar",
        "ror": "Hydro",
        "hydro": "Hydro",
        "PHS": "Hydro",
        "solid biomass": "Biomass",
        "load shedding": "load shedding",
    }
)

generator_generation = weighted_sum_timeseries(
    base_network.generators_t.p.reindex(
        columns=generators.index,
        fill_value=0,
    ),
    weights,
)

generator_generation = generator_generation.groupby(generators["carrier"]).sum()


# Storage units
if not base_network.storage_units.empty:
    storage_units = base_network.storage_units.copy()

    storage_units["carrier"] = storage_units["carrier"].replace(
        {
            "hydro": "Hydro",
            "PHS": "Hydro",
            "ror": "Hydro",
            "battery": "Battery storage",
        }
    )

    storage_generation = weighted_sum_timeseries(
        base_network.storage_units_t.p_dispatch.reindex(
            columns=storage_units.index,
            fill_value=0,
        ),
        weights,
    )

    storage_generation = storage_generation.groupby(storage_units["carrier"]).sum()

else:
    storage_generation = pd.Series(dtype=float)


# Links
links = base_network.links.copy()

links["carrier"] = links["carrier"].replace(
    {
        "OCGT": "Natural gas",
        "CCGT": "Natural gas",
        "coal": "Coal",
        "oil": "Oil",
        "biomass": "Biomass",
        "urban central solid biomass CHP": "Biomass",
        "gas CHP": "Natural gas",
        "battery discharger": "Battery storage",
    }
)

electric_buses = base_network.buses.index[
    base_network.buses.carrier.isin(["AC", "low voltage"])
]

electric_links = links[
    links.bus1.isin(electric_buses)
    & links.carrier.isin(
        [
            "Natural gas",
            "Coal",
            "Oil",
            "Biomass",
            "Battery storage",
        ]
    )
].copy()

if not electric_links.empty and hasattr(base_network.links_t, "p1"):
    link_generation = -weighted_sum_timeseries(
        base_network.links_t.p1.reindex(
            columns=electric_links.index,
            fill_value=0,
        ),
        weights,
    )

    link_generation = link_generation.clip(lower=0)

    link_generation = link_generation.groupby(electric_links["carrier"]).sum()

else:
    link_generation = pd.Series(dtype=float)


# Combine PyPSA generation and convert MWh to TWh
pypsa_generation = (
    pd.concat(
        [
            generator_generation,
            storage_generation,
            link_generation,
        ]
    )
    .groupby(level=0)
    .sum()
    / 1e6
)

# Remove non-real generation
pypsa_generation = pypsa_generation.drop(
    index=["load shedding"],
    errors="ignore",
)

pypsa_generation.name = "PyPSA-Earth (2025)"


# Rename reference carriers to match installed capacity names
reference_generation_named = reference_generation.rename(
    index={
        "battery": "Battery storage",
        "biomass": "Biomass",
        "coal": "Coal",
        "hydro": "Hydro",
        "gas": "Natural gas",
        "oil": "Oil",
        "solar": "Solar",
        "wind": "Wind",
    }
)

reference_generation_named.name = "Australian Energy Statistics (2023-24)"


# Build final comparison table
technology_order = [
    "Battery storage",
    "Biomass",
    "Coal",
    "Hydro",
    "Natural gas",
    "Oil",
    "Solar",
    "Wind",
]

annual_generation_df = (
    pd.concat(
        [
            pypsa_generation,
            reference_generation_named,
        ],
        axis=1,
    )
    .reindex(technology_order)
    .fillna(0)
)

display(annual_generation_df.round(2))

In [ ]:
# Plot comparison
ax = annual_generation_df.plot(
    kind="bar",
    figsize=(12, 6),
    width=0.8,
)

ax.set_title("Annual Electricity Generation Comparison")
ax.set_ylabel("Annual generation (TWh/year)")

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


# Total generation sanity check
print(
    f"Total PyPSA generation: "
    f"{annual_generation_df['PyPSA-Earth (2025)'].sum():.1f} TWh"
)

print(
    "Total reference generation: "
    f"{annual_generation_df['Australian Energy Statistics (2023-24)'].sum():.1f} TWh"
)

## Source:
- Australian Energy Statistics (2023-24): Department of Climate Change, Energy, the Environment and Water, Australian Energy Statistics, Table O Electricity generation by fuel type 2023-24 and 2024, 2025, Accessed May 22, 2026